# Tenacious-Bench Week 11 — CORRECT DPO Training
**Path B: DPO with Unsloth + Qwen3 + 16-bit LoRA**

Fixes from previous run:
- ✅ Uses Qwen3 (not Qwen2.5)
- ✅ 16-bit LoRA (`load_in_4bit=False`) — spec requirement
- ✅ DPO algorithm (not SFT)
- ✅ Real training log written to file
- ✅ Real held-out evaluation
- ✅ Real ablation_results.json

**Before running:** Runtime > Change runtime type > T4 GPU

In [ ]:
# CELL 1 — Install Unsloth and dependencies
# Run this, then Runtime > Restart session, then continue from Cell 2
# Installing both unsloth AND unsloth_zoo from git keeps them version-matched
%%capture
!pip install --upgrade pip -q
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" \
             "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q
!pip install trl peft transformers datasets accelerate bitsandbytes huggingface_hub -q
print('✅ Installation complete — RESTART RUNTIME NOW, then run from Cell 2')

In [ ]:
# CELL 2 — Mount Drive and check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
# T4 does NOT support bf16 → we use fp16
DTYPE = torch.float16

In [ ]:
# CELL 3 — Upload dataset files (one at a time)
import os, zipfile, json
from google.colab import files

# ── Step A: preferences_train.jsonl ──────────────────────────────────
if not os.path.exists('preferences_train.jsonl'):
    print('Upload preferences_train.jsonl')
    files.upload()
else:
    print('preferences_train.jsonl already present')

# ── Step B: preferences_dev.jsonl ────────────────────────────────────
if not os.path.exists('preferences_dev.jsonl'):
    print('Upload preferences_dev.jsonl')
    files.upload()
else:
    print('preferences_dev.jsonl already present')

# ── Step C: held_out.zip ─────────────────────────────────────────────
if not os.path.exists('held_out.zip') and not os.path.isdir('held_out'):
    print('Upload held_out.zip')
    files.upload()
else:
    print('held_out.zip / held_out/ already present')

# Extract zip if needed
if os.path.exists('held_out.zip') and not os.path.isdir('held_out'):
    with zipfile.ZipFile('held_out.zip', 'r') as z:
        z.extractall('.')
    print('Extracted held_out/')

# ── Verify ────────────────────────────────────────────────────────────
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = read_jsonl('preferences_train.jsonl')
dev_data   = read_jsonl('preferences_dev.jsonl')
held_files = [f for f in os.listdir('held_out') if f.endswith('.json')]

print(f'Train pairs : {len(train_data)}')
print(f'Dev pairs   : {len(dev_data)}')
print(f'Held-out tasks: {len(held_files)}')
print(f'Sample keys : {list(train_data[0].keys())}')

assert all(k in train_data[0] for k in ['prompt','chosen','rejected'])
assert len(held_files) == 52, f'Expected 52 held-out tasks, got {len(held_files)}'
print('✅ All files verified')

In [ ]:
# CELL 4 — Load Qwen3 with 16-bit LoRA (NO 4-bit)
# Per spec: QLoRA 4-bit is NOT used. 16-bit LoRA is the required path.
# Check Unsloth docs for latest Qwen 3.5 model name:
# https://docs.unsloth.ai/get-started/unsloth-notebooks
# Options (pick smallest that fits in VRAM at 16-bit):
#   unsloth/Qwen3-0.6B   (~1.2 GB fp16, very safe on T4)
#   unsloth/Qwen3-1.7B   (~3.4 GB fp16, fits well)
#   unsloth/Qwen3-4B     (~8 GB fp16, tight on T4)
# If Qwen 3.5 models are available in Unsloth, replace with:
#   unsloth/Qwen3.5-0.6B  or  unsloth/Qwen3.5-1.7B

from unsloth import FastLanguageModel
import torch

SEED       = 42
MODEL_NAME = "unsloth/Qwen3-1.7B"   # ← Change to Qwen3.5 if available
MAX_SEQ    = 1024

torch.manual_seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ,
    load_in_4bit   = False,          # ← REQUIRED: 16-bit, not 4-bit
    dtype          = torch.float16,  # T4 is fp16, not bf16
)

print(f'✅ Loaded {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# CELL 5 — Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    bias           = "none",
    use_gradient_checkpointing = True,
    random_state   = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({100*trainable/total:.2f}%)')
print(f'VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# CELL 6 — Prepare DPO dataset
from datasets import Dataset

def format_dpo(row):
    """DPO needs prompt / chosen / rejected as plain strings."""
    return {
        'prompt':   str(row['prompt']),
        'chosen':   str(row['chosen']),
        'rejected': str(row['rejected']),
    }

train_ds = Dataset.from_list([format_dpo(r) for r in train_data])
dev_ds   = Dataset.from_list([format_dpo(r) for r in dev_data])

print(f'Train: {len(train_ds)} | Dev: {len(dev_ds)}')
print('Sample prompt[:150]:', train_ds[0]['prompt'][:150])
print('Chosen[:100]:', train_ds[0]['chosen'][:100])
print('Rejected[:100]:', train_ds[0]['rejected'][:100])

In [ ]:
# CELL 7 — DPO Training (Path B per spec)
# Patch Unsloth into TRL's DPO trainer for 2x speed
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig
import time, datetime

OUTPUT_DIR = '/content/drive/MyDrive/tenacious-dpo-v0.1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

dpo_config = DPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,       # effective batch = 8
    learning_rate               = 2e-5,
    beta                        = 0.1,     # DPO beta — conservative (small dataset)
    max_prompt_length           = 512,
    max_length                  = 1024,
    fp16                        = True,    # T4 is fp16
    bf16                        = False,
    seed                        = SEED,
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = 20,
    save_steps                  = 60,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    report_to                   = 'none',  # no wandb
    remove_unused_columns       = False,
)

trainer = DPOTrainer(
    model           = model,
    ref_model       = None,  # Unsloth handles implicit reference
    args            = dpo_config,
    train_dataset   = train_ds,
    eval_dataset    = dev_ds,
    tokenizer       = tokenizer,
)

print(f'Starting DPO training at {datetime.datetime.utcnow().isoformat()}Z')
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
print(f'✅ Training done in {elapsed/60:.1f} min')
print(f'Final loss: {train_result.training_loss:.4f}')
print(f'Steps: {train_result.global_step}')

In [ ]:
# CELL 8 — Plot loss curve
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps  = [x['step'] for x in history if 'loss' in x]
train_losses = [x['loss'] for x in history if 'loss' in x]
eval_steps   = [x['step'] for x in history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in history if 'eval_loss' in x]

reward_margins = [x.get('rewards/margins', None) for x in history if 'rewards/margins' in x]
reward_steps   = [x['step'] for x in history if 'rewards/margins' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_steps, train_losses, label='train loss')
if eval_losses: ax1.plot(eval_steps, eval_losses, label='eval loss')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('DPO Loss')

if reward_margins:
    ax2.plot(reward_steps, reward_margins, color='green')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Reward Margin'); ax2.set_title('DPO Reward Margin')

plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print('Saved loss_curve.png')

In [ ]:
# CELL 9 — Write REAL training log (from actual trainer.state)
import json, datetime, os

history = trainer.state.log_history
loss_curve = [(x['step'], round(x['loss'],4), round(x.get('eval_loss', x['loss']),4))
              for x in history if 'loss' in x]

log_lines = [
    f"=== Tenacious-Bench DPO Training Run ===",
    f"Model:      {MODEL_NAME}",
    f"Adapter:    LoRA r=16, alpha=32",
    f"Algorithm:  DPO (Rafailov et al., NeurIPS 2023)",
    f"Hardware:   Google Colab T4 (free tier)",
    f"Seed:       {SEED}",
    f"Started:    {datetime.datetime.utcnow().isoformat()}Z",
    f"Wall time:  {elapsed/60:.1f} min",
    f"",
    f"=== Hyperparameters ===",
    f"learning_rate:    {dpo_config.learning_rate}",
    f"beta (DPO):       {dpo_config.beta}",
    f"batch_size:       {dpo_config.per_device_train_batch_size} x {dpo_config.gradient_accumulation_steps} = {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps} effective",
    f"epochs:           {dpo_config.num_train_epochs}",
    f"total_steps:      {train_result.global_step}",
    f"precision:        fp16 (T4 native)",
    f"quantization:     None (16-bit LoRA)",
    f"",
    f"=== Training Data ===",
    f"train_file:   preferences_train.jsonl",
    f"train_pairs:  {len(train_ds)}",
    f"dev_pairs:    {len(dev_ds)}",
    f"",
    f"=== Loss Curve ===",
]
for step, tl, el in loss_curve:
    log_lines.append(f"Step {step:4d} | train_loss: {tl:.4f} | eval_loss: {el:.4f}")

log_lines += [
    f"",
    f"=== Final Metrics ===",
    f"final_train_loss:  {train_result.training_loss:.4f}",
    f"total_steps:       {train_result.global_step}",
    f"samples_per_sec:   {train_result.metrics.get('train_samples_per_second', 'N/A')}",
]

log_text = '\n'.join(log_lines)
log_path = f'training_run_seed{SEED}.log'
with open(log_path, 'w') as f:
    f.write(log_text)

print(log_text)
print(f'\n✅ Saved to {log_path}')

In [ ]:
# CELL 10 — Push LoRA adapter to HuggingFace
# Set your HF token first:
HF_TOKEN    = "hf_YOUR_TOKEN_HERE"   # ← paste your write token
HF_REPO     = "meseretbolled/Tenacious-Qwen3-DPO-v01"  # ← your HF username/repo

from huggingface_hub import login
login(token=HF_TOKEN)

# Save adapter only (not full model)
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f'✅ Adapter pushed to huggingface.co/{HF_REPO}')

In [ ]:
# CELL 11 — Held-out evaluation (REAL scoring)
# Scores both base model and trained model on all 52 held-out tasks
# Uses rule-based dimensions (no external API needed for core scoring)

import os, json, re, glob
from transformers import AutoModelForCausalLM, AutoTokenizer

FastLanguageModel.for_inference(model)  # Enable 2x faster inference

BANNED_PHRASES = [
    r'\bbench\b', r'world.class', r'ninja', r'rockstar', r'a.players',
    r'top.tier', r'elite', r'unicorn', r'guru', r'wizard',
    r'thought.leader', r'synergy'
]

def generate(mdl, tok, prompt_text, max_new=300):
    inputs = tok(prompt_text, return_tensors='pt', truncation=True,
                 max_length=512).to('cuda')
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new,
                           temperature=0.1, do_sample=True,
                           pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def score_output(task, output_text):
    scores = {}
    rubric  = task.get('expected_rubric', {})
    brief   = task.get('input', {}).get('hiring_signal_brief', {})
    bench   = task.get('input', {}).get('bench_state', {})

    # 1. Banned phrase check (weight 0.10)
    scores['banned_phrase_check'] = int(
        not any(re.search(p, output_text, re.I) for p in BANNED_PHRASES)
    )

    # 2. Booking link present (weight 0.10)
    scores['booking_link_present'] = int('cal.com' in output_text or
                                         'calendly' in output_text.lower())

    # 3. Signal confidence compliance (weight 0.25)
    phrasing_mode = brief.get('ai_maturity', {}).get('phrasing_mode', 'observe')
    if phrasing_mode == 'ask':
        scores['signal_confidence_compliance'] = int('?' in output_text)
    else:
        scores['signal_confidence_compliance'] = 1

    # 4. ICP segment correctness (weight 0.20) — check expected segment keywords
    expected_seg = task.get('ground_truth_segment', '')
    company = brief.get('company', '')
    scores['icp_segment_correctness'] = int(
        company.lower() in output_text.lower()
    )

    # 5. Bench capacity honesty (weight 0.20, hard gate)
    total_bench = sum(bench.values()) if bench else 0
    # Check if output claims more than total available
    numbers_in_output = re.findall(r'\b(\d{1,3})\b', output_text)
    over_committed = any(int(n) > total_bench * 1.5 and int(n) > 5
                         for n in numbers_in_output if total_bench > 0)
    scores['bench_capacity_honesty'] = int(not over_committed)

    # 6. Tone compliance (weight 0.15) — simplified rule-based version
    # Full version uses LLM judge; this uses proxy rules
    # Fail if: all-caps words, excessive exclamation, condescending phrases
    exclamation_count  = output_text.count('!')
    condescending      = bool(re.search(r"you clearly|obviously you|don't worry", output_text, re.I))
    scores['tone_compliance'] = int(exclamation_count <= 2 and not condescending)

    # Weighted score
    weights = {
        'signal_confidence_compliance': 0.25,
        'icp_segment_correctness':      0.20,
        'bench_capacity_honesty':       0.20,
        'tone_compliance':              0.15,
        'booking_link_present':         0.10,
        'banned_phrase_check':          0.10,
    }
    weighted = sum(scores[k] * weights[k] for k in weights)
    scores['weighted_score'] = round(weighted, 3)
    scores['overall_pass']   = weighted >= 0.70

    # Hard gate: bench capacity failure is automatic fail
    if scores['bench_capacity_honesty'] == 0:
        scores['weighted_score'] = 0.0
        scores['overall_pass']   = False

    return scores

def task_to_prompt(task):
    brief   = task.get('input', {}).get('hiring_signal_brief', {})
    prospect = task.get('input', {}).get('prospect_context', {})
    bench   = task.get('input', {}).get('bench_state', {})
    company = brief.get('company', 'the company')
    name    = prospect.get('name', 'the prospect')
    title   = prospect.get('title', prospect.get('role', 'VP Engineering'))
    return (
        f"Write a professional B2B sales outreach email from Tenacious Intelligence.\n"
        f"Prospect: {name}, {title} at {company}.\n"
        f"Available bench: {json.dumps(bench)}.\n"
        f"Context: {json.dumps(brief)}.\n"
        f"Requirements: professional tone, no banned jargon, honest capacity, "
        f"include calendar link https://cal.com/meseret-bolled-pxprep/tenacious-discovery-call"
    )

# Load base model for comparison
print('Loading base model for comparison...')
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name   = MODEL_NAME,
    max_seq_length = MAX_SEQ,
    load_in_4bit   = False,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(base_model)
print('✅ Base model loaded')

In [ ]:
# CELL 12 — Run held-out evaluation
held_out_dir = 'held_out/'
task_files   = sorted(glob.glob(os.path.join(held_out_dir, '*.json')))
print(f'Evaluating {len(task_files)} held-out tasks...')

traces       = []
base_scores  = []
train_scores = []

for i, tf in enumerate(task_files):
    task = json.load(open(tf))
    prompt_text = task_to_prompt(task)

    base_out    = generate(base_model, base_tok, prompt_text)
    trained_out = generate(model, tokenizer, prompt_text)

    base_sc    = score_output(task, base_out)
    trained_sc = score_output(task, trained_out)

    base_scores.append(base_sc['weighted_score'])
    train_scores.append(trained_sc['weighted_score'])

    traces.append({
        'task_id':             task.get('task_id', os.path.basename(tf)),
        'company':             task.get('input',{}).get('hiring_signal_brief',{}).get('company',''),
        'partition':           'held_out',
        'base_model_output':   base_out,
        'trained_model_output': trained_out,
        'scoring': {
            'base_model':    base_sc,
            'trained_model': trained_sc,
        }
    })

    if (i+1) % 10 == 0:
        print(f'  [{i+1}/{len(task_files)}] base_mean={sum(base_scores)/len(base_scores):.3f} \
trained_mean={sum(train_scores)/len(train_scores):.3f}')

base_mean  = sum(base_scores) / len(base_scores)
train_mean = sum(train_scores) / len(train_scores)
improvement = train_mean - base_mean

print(f'\n=== RESULTS ===')
print(f'Base mean:    {base_mean:.3f}')
print(f'Trained mean: {train_mean:.3f}')
print(f'Delta A:      {improvement:+.3f}')

In [ ]:
# CELL 13 — Bootstrap confidence interval + p-value
import numpy as np

np.random.seed(SEED)
diffs = np.array(train_scores) - np.array(base_scores)
N_BOOT = 10000
boot_means = [np.mean(np.random.choice(diffs, size=len(diffs), replace=True))
              for _ in range(N_BOOT)]

ci_lo = float(np.percentile(boot_means, 2.5))
ci_hi = float(np.percentile(boot_means, 97.5))
p_val = float(np.mean(np.array(boot_means) <= 0.0))  # one-tailed

print(f'Delta A: {improvement:+.4f}')
print(f'95% CI:  [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'p-value: {p_val:.4f} (one-tailed, H0: improvement <= 0)')
print(f'Significant (p<0.05): {p_val < 0.05}')

In [ ]:
# CELL 14 — Write held_out_traces.jsonl and ablation_results.json
import datetime

# Write ALL traces (not just 17)
with open('held_out_traces.jsonl', 'w') as f:
    for t in traces:
        f.write(json.dumps(t) + '\n')
print(f'Wrote {len(traces)} traces to held_out_traces.jsonl')

# Write ablation results
ablation = {
    'project':          'Tenacious-Bench Week 11',
    'run_date':         datetime.date.today().isoformat(),
    'seed':             SEED,
    'base_model':       MODEL_NAME,
    'trained_model_id': f'huggingface.co/{HF_REPO}',
    'training_algorithm': 'DPO',
    'held_out_tasks':   len(task_files),
    'delta_a': {
        'base_score':           round(base_mean, 4),
        'trained_score':        round(train_mean, 4),
        'absolute_improvement': round(improvement, 4),
        'relative_improvement_pct': round(100 * improvement / max(base_mean, 1e-6), 2),
        'n_pairs':              len(task_files),
        'bootstrap_mean_diff':  round(float(np.mean(boot_means)), 4),
        'bootstrap_std':        round(float(np.std(boot_means)), 4),
        'ci_95_lower':          round(ci_lo, 4),
        'ci_95_upper':          round(ci_hi, 4),
        'p_value':              round(p_val, 4),
        'significant':          p_val < 0.05,
    },
    'per_task_base':    base_scores,
    'per_task_trained': train_scores,
}

with open('ablation_results.json', 'w') as f:
    json.dump(ablation, f, indent=2)
print('Wrote ablation_results.json')

# Summary
print(f'\n=== FINAL ABLATION SUMMARY ===')
print(f'Base:      {base_mean:.4f}')
print(f'Trained:   {train_mean:.4f}')
print(f'Delta A:   {improvement:+.4f}')
print(f'95% CI:    [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'p-value:   {p_val:.4f}')

In [ ]:
# CELL 15 — Download output files
from google.colab import files
import os

for fname in ['ablation_results.json', 'held_out_traces.jsonl',
              f'training_run_seed{SEED}.log', 'loss_curve.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')
    else:
        print(f'MISSING: {fname}')

print('\n✅ Copy to repo:')
print('  ablation_results.json  → tenacious-bench/')
print('  held_out_traces.jsonl  → tenacious-bench/')
print('  training_run_seed42.log → tenacious-bench/training/')
print('  loss_curve.png          → tenacious-bench/training/')
